# 04 · What it cost to run

The cost ledger only exists on the current platform, and only for live traffic — history that was
imported from the old database carries no cost rows. So this is a **Fall 2026 figure**, over
whatever window the ledger actually covers, and the report states that window rather than
extrapolating a per-student cost across a year that was never metered.

If the ledger is empty the notebook still runs and the report simply omits the cost slide.

In [1]:
# Put the analysis package on the path no matter where Jupyter was started.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "analysis" / "bloombot_analysis").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "analysis"))

import pandas as pd

from bloombot_analysis import charts, load, metrics, privacy, report, sessions, topics
from bloombot_analysis.config import CONFIG, SURFACE_LABELS, TOPICS

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("as of:", CONFIG.as_of)
print("legacy db :", CONFIG.legacy_db, "(exists)" if CONFIG.legacy_db.exists() else "(missing)")
print("current db:", CONFIG.current_db, "(exists)" if CONFIG.current_db.exists() else "(missing)")
print("output    :", CONFIG.out_dir)

as of: 2026-09-25
legacy db : /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/legacy.db (exists)
current db: /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/current.db (exists)
output    : /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/out


In [2]:
sessions_df = pd.read_csv(
    CONFIG.data_path("sessions.csv"), parse_dates=["started_at", "ended_at", "week"]
)
sessions_df["date"] = pd.to_datetime(sessions_df["date"]).dt.date
messages_df = pd.read_csv(CONFIG.data_path("messages.csv"), parse_dates=["ts", "week"])
print(len(sessions_df), "sessions,", len(messages_df), "messages")

259 sessions, 1584 messages


In [3]:
costs_df = load.load_costs()
if costs_df.empty:
    print("no cost ledger rows — the report will skip the cost slide")
else:
    print(
        f"{len(costs_df):,} ledger rows, "
        f"{costs_df['ts'].min().date()} → {costs_df['ts'].max().date()}, "
        f"${costs_df['usd'].sum():,.2f} total"
    )
costs_df.head()

104 ledger rows, 2026-09-02 → 2026-09-24, $0.74 total


,ts,course,surface,model,input_tokens,output_tokens,usd
0,2026-09-02 01:55:00,Introduction to Programming,discord,gpt-4.1,2921,422,0.009218
1,2026-09-02 02:09:23,Introduction to Programming,discord,gpt-4.1,2665,300,0.007730
2,2026-09-02 02:11:12,Introduction to Programming,discord,gpt-4.1,2915,499,0.009822
3,2026-09-02 02:19:33,Introduction to Programming,discord,gpt-4.1,2204,682,0.009864
4,2026-09-02 02:27:34,Introduction to Programming,discord,gpt-4.1,1033,269,0.004218


In [4]:
if costs_df.empty:
    cost_summary = {"rows": 0}
    cost_fig = None
    by_course = pd.DataFrame()
else:
    covered = sessions_df[
        (sessions_df["started_at"] >= costs_df["ts"].min())
        & (sessions_df["started_at"] <= costs_df["ts"].max())
    ]
    by_course = (
        costs_df.groupby("course")
        .agg(usd=("usd", "sum"), calls=("usd", "count"),
             input_tokens=("input_tokens", "sum"), output_tokens=("output_tokens", "sum"))
        .reset_index()
        .sort_values("usd", ascending=False)
    )
    cost_fig = charts.bar_h(
        by_course["course"], by_course["usd"], "cost_by_course",
        "Model spend by course (ledger window only)", "US dollars",
    )
    cost_summary = {
        "rows": int(len(costs_df)),
        "window_start": costs_df["ts"].min(),
        "window_end": costs_df["ts"].max(),
        "total_usd": float(costs_df["usd"].sum()),
        "sessions_in_window": int(len(covered)),
        "students_in_window": int(covered["person_key"].nunique()) if len(covered) else 0,
        "usd_per_session": float(costs_df["usd"].sum() / len(covered)) if len(covered) else None,
        "usd_per_student": (
            float(costs_df["usd"].sum() / covered["person_key"].nunique())
            if len(covered) and covered["person_key"].nunique()
            else None
        ),
        "models": costs_df["model"].value_counts().to_dict(),
        "by_course": by_course.to_dict(orient="records"),
        "figure": cost_fig.name if cost_fig is not None else None,
    }
cost_summary

{'rows': 104,
 'window_start': Timestamp('2026-09-02 01:55:00'),
 'window_end': Timestamp('2026-09-24 09:10:00'),
 'total_usd': 0.740448,
 'sessions_in_window': 41,
 'students_in_window': 24,
 'usd_per_session': 0.01805970731707317,
 'usd_per_student': 0.030852,
 'models': {'gpt-4.1': 104},
 'by_course': [{'course': 'Introduction to Programming',
   'usd': 0.318236,
   'calls': 42,
   'input_tokens': 87074,
   'output_tokens': 18011},
  {'course': 'Web Design',
   'usd': 0.193526,
   'calls': 28,
   'input_tokens': 52419,
   'output_tokens': 11086},
  {'course': 'Software Engineering',
   'usd': 0.14037,
   'calls': 20,
   'input_tokens': 43213,
   'output_tokens': 6743},
  {'course': 'Agile Software Development & DevOps',
   'usd': 0.088316,
   'calls': 14,
   'input_tokens': 24338,
   'output_tokens': 4955}],
 'figure': 'cost_by_course.png'}

In [5]:
metrics.update("cost", cost_summary)
print("ok")

ok
